# LLaMA 1B x MATH — BF16 — All 50 Problems

**Quantization:** BF16 full precision

**Problems:** 50 total (20 Level 1, 20 Level 3, 10 Level 5)

**Strategies:** Greedy + Self-Consistency N=5

**Output:** `llama_math_bf16.csv`

**Estimated time:** ~140-170 min on T4


In [1]:
# ── CELL 1: Install ───────────────────────────────────────────────────────────
!pip install -q "transformers>=4.40.0" "datasets>=2.19.0" "accelerate>=0.29.0" "bitsandbytes>=0.43.0"
print('Done')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00
Dependencies installed


In [2]:
# ── CELL 2: HuggingFace login — required for gated LLaMA ──────────────────────
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))
print('Logged in via secret')


Logged in via hardcoded token


In [3]:
# ── CELL 3: GPU check ─────────────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU — Runtime > Change runtime type > T4 GPU')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


CUDA: True
GPU:  Tesla T4
VRAM: 15.6 GB


In [4]:
# ── CELL 4: Imports ───────────────────────────────────────────────────────────
import re, gc, random, time, os
import torch, pandas as pd
from collections import Counter
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
print('Imports OK')


Imports OK


In [6]:
# ── CELL 5: Config ────────────────────────────────────────────────────────────
MODEL_ID       = 'meta-llama/Llama-3.2-1B-Instruct'
QUANT_TYPE     = 'bf16'
OUTPUT_PATH    = '/content/llama_math_bf16.csv'
SEED           = 42
MAX_NEW_TOKENS = 1024
SC_N           = 5
SC_TEMP        = 0.7

random.seed(SEED)
torch.manual_seed(SEED)
print(f'Quant: {QUANT_TYPE} | Output: {OUTPUT_PATH}')


Account:    2 of 4
Quant type: bf16
Output:     /content/llama_math_account2.csv
Problems:   13 (5 L1, 5 L3, 3 L5)


In [7]:
# ── CELL 6: Load MATH dataset — all 50 problems ───────────────────────────────
# Same dataset, seed, and sampling as the other LLaMA MATH runs so problem IDs match

print('Loading MATH dataset...')
subjects = ['algebra', 'geometry', 'number_theory', 'prealgebra', 'precalculus']
datasets_list = []
for sub in subjects:
    print(f'  Loading {sub}...')
    datasets_list.append(load_dataset('EleutherAI/hendrycks_math', sub, split='test'))
ds_math = concatenate_datasets(datasets_list)

random.seed(SEED)
easy_all, medium_all, hard_all = [], [], []
for i, row in enumerate(ds_math):
    level = row['level']
    if '1' in level:
        easy_all.append(i)
    elif '3' in level:
        medium_all.append(i)
    elif '5' in level:
        hard_all.append(i)

random.shuffle(easy_all)
random.shuffle(medium_all)
random.shuffle(hard_all)
all_l1 = easy_all[:20]
all_l3 = medium_all[:20]
all_l5 = hard_all[:10]

problems = []
for idx in all_l1:
    row = ds_math[idx]
    problems.append({'pid': idx, 'difficulty': 'Level 1',
                     'question': row['problem'], 'answer': row['solution']})
for idx in all_l3:
    row = ds_math[idx]
    problems.append({'pid': idx, 'difficulty': 'Level 3',
                     'question': row['problem'], 'answer': row['solution']})
for idx in all_l5:
    row = ds_math[idx]
    problems.append({'pid': idx, 'difficulty': 'Level 5',
                     'question': row['problem'], 'answer': row['solution']})

print(f'Loaded {len(problems)} problems total')
print('Level 1:', sum(p['difficulty'] == 'Level 1' for p in problems))
print('Level 3:', sum(p['difficulty'] == 'Level 3' for p in problems))
print('Level 5:', sum(p['difficulty'] == 'Level 5' for p in problems))


Loading MATH dataset...
  Loading algebra...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

algebra/train-00000-of-00001.parquet:   0%|          | 0.00/505k [00:00<?, ?B/s]

algebra/test-00000-of-00001.parquet:   0%|          | 0.00/353k [00:00<?, ?B/s]

...

precalculus/train-00000-of-00001.parquet:   0%|          | 0.00/354k [00:00<?, ?B/s]

precalculus/test-00000-of-00001.parquet:   0%|          | 0.00/242k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/746 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/546 [00:00<?, ? examples/s]

Total MATH test problems: 3623
Level 1 available: 346
Level 3 available: 836
Level 5 available: 921

This account: 13 problems
  Level 1: 5
  Level 3: 5
  Level 5: 3
Example: In $\triangle ABC$, what is the value of $x + y$? [asy]
import olympiad;
size(7cm);

pair a = dir(76);
pair b = (-1, 0);...


In [8]:
# ── CELL 7: Helpers ───────────────────────────────────────────────────────────

def extract_ground_truth(solution_str):
    matches = re.findall(r'\\boxed\{([^}]+)\}', solution_str)
    if matches: return matches[-1].strip()
    nums = re.findall(r'-?[\d\.]+', solution_str)
    return nums[-1] if nums else None

def extract_prediction(output_str):
    matches = re.findall(r'\\boxed\{([^}]+)\}', output_str)
    if matches: return matches[-1].strip()
    m = re.search(r'####\s*([\d,\.\-]+)', output_str)
    if m: return m.group(1).replace(',', '').strip()
    nums = re.findall(r'\b([\d,\.]+)\b', output_str)
    if nums: return nums[-1].replace(',', '').strip()
    return None

def format_prompt(question):
    return (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
        'You are a math problem solver. Show your reasoning step by step. '
        'End your answer with the final result in \\boxed{}.\n'
        '<|eot_id|><|start_header_id|>user<|end_header_id|>\n'
        f'{question}\n'
        '<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n'
    )

def detect_first_error_step(text):
    steps = re.split(r'\n+', text)
    for step_num, step in enumerate(steps, 1):
        matches = re.findall(r'([\d\.]+)\s*([\+\-\*\/])\s*([\d\.]+)\s*=\s*([\d\.]+)', step)
        for a, op, b, result in matches:
            try:
                a, b, result = float(a), float(b), float(result)
                expected = {'+': a+b, '-': a-b, '*': a*b, '/': a/b if b != 0 else None}.get(op)
                if expected is not None and abs(expected - result) > 0.01:
                    return step_num
            except Exception:
                continue
    return None

print('GT test:  ', extract_ground_truth('Therefore $\\boxed{42}$.'))
print('Pred test:', extract_prediction('The answer is $\\boxed{42}$.'))



GT test:    42
Pred test:  42


In [17]:
# ── CELL 8: Load model BF16 ──────────────────────────────────────────────────

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

print('Loading model BF16...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True
)
model.eval()
print(f'GPU mem after load: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print('Model ready')


Loading tokenizer (meta-llama/Llama-3.2-1B-Instruct)...
Loading model in bf16...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

GPU mem after load: 2.48 GB
Model ready


In [10]:
# ── CELL 9: Run greedy — all 50 problems ──────────────────────────────────────
# Saves after every problem — resume-safe

existing    = pd.read_csv(OUTPUT_PATH) if os.path.exists(OUTPUT_PATH) else pd.DataFrame()
done_greedy = set(existing[existing['decoding_strategy']=='greedy']['pid'].tolist())               if len(existing) > 0 else set()
greedy_results = []

print('='*55)
print(f'GREEDY — BF16 — {len(problems)} problems')
print('='*55)
t0 = time.time()

for i, prob in enumerate(problems):
    if prob['pid'] in done_greedy:
        print(f'  [{i+1}/{len(problems)}] pid={prob["pid"]} already done, skipping')
        continue

    gt     = extract_ground_truth(prob['answer'])
    prompt = format_prompt(prob['question'])
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = tokenizer.decode(
        out_ids[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    pred    = extract_prediction(generated)
    correct = (pred is not None and gt is not None and pred.strip() == gt.strip())
    err     = detect_first_error_step(generated) if not correct else None
    elapsed = (time.time() - t0) / 60

    greedy_results.append({
        'quant_type':              QUANT_TYPE,
        'decoding_strategy':       'greedy',
        'pid':                     prob['pid'],
        'difficulty':              prob['difficulty'],
        'predicted_answer':        pred,
        'ground_truth':            gt,
        'correct':                 correct,
        'reasoning_length_tokens': len(out_ids[0]) - inputs['input_ids'].shape[1],
        'first_error_step':        err,
        'raw_output':              generated[:500],
    })

    all_rows = pd.concat([existing, pd.DataFrame(greedy_results)], ignore_index=True)
    all_rows.to_csv(OUTPUT_PATH, index=False)

    status = 'CORRECT' if correct else 'wrong'
    print(f'  [{i+1}/{len(problems)}] pid={prob["pid"]:<6} '
          f'{prob["difficulty"]:<8} '
          f'pred={str(pred):<12} gt={str(gt):<12} '
          f'{status}  err={err}  {elapsed:.0f}min')

n_correct = sum(r['correct'] for r in greedy_results)
print('='*55)
print(f'Greedy done: {n_correct}/{len(greedy_results)} '
      f'({100*n_correct/max(len(greedy_results),1):.1f}%)  '
      f'{(time.time()-t0)/60:.1f} min')


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


GREEDY — Account 2 of 4 | bf16
  [1/13] pid=1575   Level 1  pred=76           gt=90           wrong  err=None  0min
  [2/13] pid=3561   Level 1  pred=\frac{7      gt=\frac{24     wrong  err=None  0min
  [3/13] pid=673    Level 1  pred=2000         gt=2000         CORRECT  err=None  0min
  [4/13] pid=3291   Level 1  pred=72           gt=72           CORRECT  err=None  1min
  [5/13] pid=235    Level 1  pred=108          gt=36           wrong  err=None  1min
  [6/13] pid=2998   Level 3  pred=3            gt=9            wrong  err=None  1min
  [7/13] pid=2571   Level 3  pred=35           gt=\frac{4      wrong  err=None  1min
  [8/13] pid=941    Level 3  pred=25           gt=25           CORRECT  err=None  1min
  [9/13] pid=509    Level 3  pred=105          gt=17           wrong  err=None  1min
  [10/13] pid=60     Level 3  pred=(7, -46)     gt=7            wrong  err=None  1min
  [11/13] pid=2890   Level 5  pred=1            gt=36           wrong  err=None  2min
  [12/13] pid=756    Level

In [11]:
# ── CELL 10: Run SC N=5 — all 50 problems ─────────────────────────────────────
# Saves after every problem

existing2 = pd.read_csv(OUTPUT_PATH) if os.path.exists(OUTPUT_PATH) else pd.DataFrame()
done_sc   = set(existing2[existing2['decoding_strategy']=='self_consistency']['pid'].tolist())             if len(existing2) > 0 else set()
sc_results = []

print('='*55)
print(f'SELF-CONSISTENCY N=5 — BF16 — {len(problems)} problems')
print('='*55)
t0 = time.time()

for i, prob in enumerate(problems):
    if prob['pid'] in done_sc:
        print(f'  [{i+1}/{len(problems)}] pid={prob["pid"]} already done, skipping')
        continue

    gt     = extract_ground_truth(prob['answer'])
    prompt = format_prompt(prob['question'])
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    path_preds = []
    for _ in range(SC_N):
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=SC_TEMP,
                pad_token_id=tokenizer.eos_token_id
            )
        gen = tokenizer.decode(
            out_ids[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
        p = extract_prediction(gen)
        if p is not None:
            path_preds.append(p.strip())

    final_pred = Counter(path_preds).most_common(1)[0][0] if path_preds else None
    correct    = (final_pred is not None and gt is not None and final_pred.strip() == gt.strip())
    elapsed    = (time.time() - t0) / 60

    sc_results.append({
        'quant_type':              QUANT_TYPE,
        'decoding_strategy':       'self_consistency',
        'pid':                     prob['pid'],
        'difficulty':              prob['difficulty'],
        'predicted_answer':        final_pred,
        'ground_truth':            gt,
        'correct':                 correct,
        'reasoning_length_tokens': SC_N,
        'first_error_step':        None,
        'raw_output':              str(path_preds),
        'n_samples':               SC_N,
    })

    all_rows2 = pd.concat([existing2, pd.DataFrame(sc_results)], ignore_index=True)
    all_rows2.to_csv(OUTPUT_PATH, index=False)

    status = 'CORRECT' if correct else 'wrong'
    print(f'  [{i+1}/{len(problems)}] pid={prob["pid"]:<6} '
          f'{prob["difficulty"]:<8} '
          f'vote={str(final_pred):<12} gt={str(gt):<12} '
          f'{status}  paths={len(path_preds)}/{SC_N}  {elapsed:.0f}min')

n_correct_sc = sum(r['correct'] for r in sc_results)
print('='*55)
print(f'SC done: {n_correct_sc}/{len(sc_results)} '
      f'({100*n_correct_sc/max(len(sc_results),1):.1f}%)  '
      f'{(time.time()-t0)/60:.1f} min')


SELF-CONSISTENCY N=5 — Account 2 | bf16
  [1/13] pid=1575   Level 1  vote=76           gt=90           wrong  paths=5/5  1min
  [2/13] pid=3561   Level 1  vote=\frac{6      gt=\frac{24     wrong  paths=5/5  2min
  [3/13] pid=673    Level 1  vote=2000         gt=2000         CORRECT  paths=5/5  2min
  [4/13] pid=3291   Level 1  vote=72           gt=72           CORRECT  paths=5/5  3min
  [5/13] pid=235    Level 1  vote=36           gt=36           CORRECT  paths=5/5  4min
  [6/13] pid=2998   Level 3  vote=27           gt=9            wrong  paths=5/5  4min
  [7/13] pid=2571   Level 3  vote=\frac{4      gt=\frac{4      CORRECT  paths=5/5  5min
  [8/13] pid=941    Level 3  vote=25           gt=25           CORRECT  paths=5/5  5min
  [9/13] pid=509    Level 3  vote=126          gt=17           wrong  paths=5/5  6min
  [10/13] pid=60     Level 3  vote=7            gt=7            CORRECT  paths=5/5  7min
  [11/13] pid=2890   Level 5  vote=18           gt=36           wrong  paths=5/5  7min


In [12]:
# ── CELL 11: Summary + save to Drive ──────────────────────────────────────────
final_df = pd.read_csv(OUTPUT_PATH)
print(f'Total rows: {len(final_df)}')
for strat in final_df['decoding_strategy'].unique():
    sub = final_df[final_df['decoding_strategy']==strat]
    print(f'{strat}: {sub["correct"].mean()*100:.1f}% ({int(sub["correct"].sum())}/{len(sub)})')
    for diff in ['Level 1','Level 3','Level 5']:
        d = sub[sub['difficulty']==diff]
        if len(d) > 0:
            print(f'  {diff}: {d["correct"].mean()*100:.1f}% ({int(d["correct"].sum())}/{len(d)})')

print('\nDOWNLOAD: llama_math_bf16.csv from Files panel (left sidebar → folder icon)')


Total rows saved: 26

greedy: 30.8% (4/13)
  Level 1: 40.0% (2/5)
  Level 3: 20.0% (1/5)
  Level 5: 33.3% (1/3)
self_consistency: 53.8% (7/13)
  Level 1: 60.0% (3/5)
  Level 3: 60.0% (3/5)
  Level 5: 33.3% (1/3)

DOWNLOAD: llama_math_account2.csv from Files panel (left sidebar → folder icon)
Change QUANT_TYPE in Cell 5 and restart runtime for next quantization level.


In [13]:
# ── CELL 12: Cleanup ──────────────────────────────────────────────────────────
del model
gc.collect()
torch.cuda.empty_cache()
print(f'GPU mem after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB')

GPU mem after cleanup: 0.01 GB


In [14]:
# Disabled: legacy per-account cell retained from parallel runs.
print('Skipping legacy account-specific cell 13; use Cells 5–11 for the full 50-problem run.')



Account: 1
Output:  /content/llama_math_account1.csv


In [15]:
# Disabled: legacy per-account cell retained from parallel runs.
print('Skipping legacy account-specific cell 14; use Cells 5–11 for the full 50-problem run.')



Account 1 problems loaded: 13
  Level 1: 5
  Level 3: 5
  Level 5: 3


In [18]:
# Disabled: legacy per-account cell retained from parallel runs.
print('Skipping legacy account-specific cell 15; use Cells 5–11 for the full 50-problem run.')



GREEDY — Account 1 of 4 | bf16
  [1/13] pid=2054   Level 1  pred=7            gt=7            CORRECT  err=None  0min
  [2/13] pid=399    Level 1  pred=4            gt=4            CORRECT  err=None  0min
  [3/13] pid=925    Level 1  pred=18           gt=18           CORRECT  err=None  0min
  [4/13] pid=922    Level 1  pred=1.32         gt=50           wrong  err=None  0min
  [5/13] pid=1343   Level 1  pred=90           gt=90^\circ     wrong  err=None  1min
  [6/13] pid=3194   Level 3  pred=1            gt=\frac{7      wrong  err=None  1min
  [7/13] pid=3388   Level 3  pred=3 \sqrt{5    gt=3 \sqrt{5    CORRECT  err=None  1min
  [8/13] pid=911    Level 3  pred=-2           gt=9            wrong  err=None  1min
  [9/13] pid=1307   Level 3  pred=759.75\pi    gt=525\pi       wrong  err=None  2min
  [10/13] pid=2522   Level 3  pred=80           gt=80           CORRECT  err=None  2min
  [11/13] pid=3211   Level 5  pred=7            gt=\left( \frac{7 wrong  err=None  2min
  [12/13] pid=1931  

In [19]:
# Disabled: legacy per-account cell retained from parallel runs.
print('Skipping legacy account-specific cell 16; use Cells 5–11 for the full 50-problem run.')



SELF-CONSISTENCY N=5 — Account 1 | bf16


/tmp/ipykernel_2715/1292921251.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_rows2 = pd.concat([existing2, pd.DataFrame(sc_results)], ignore_index=True)


  [1/13] pid=2054   Level 1  vote=7            gt=7            CORRECT  paths=5/5  0min


/tmp/ipykernel_2715/1292921251.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_rows2 = pd.concat([existing2, pd.DataFrame(sc_results)], ignore_index=True)


  [2/13] pid=399    Level 1  vote=4            gt=4            CORRECT  paths=5/5  1min


...

/tmp/ipykernel_2715/1292921251.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_rows2 = pd.concat([existing2, pd.DataFrame(sc_results)], ignore_index=True)


  [11/13] pid=3211   Level 5  vote=9            gt=\left( \frac{7 wrong  paths=5/5  12min


/tmp/ipykernel_2715/1292921251.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_rows2 = pd.concat([existing2, pd.DataFrame(sc_results)], ignore_index=True)


  [12/13] pid=1931   Level 5  vote=(2,54), (3,36), (4,27), (6,18), (9,12) gt=4            wrong  paths=5/5  13min
  [13/13] pid=1396   Level 5  vote=5            gt=3+ \frac{\sqrt{69 wrong  paths=5/5  14min
SC done: 4/13 (30.8%)  14.2 min


/tmp/ipykernel_2715/1292921251.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_rows2 = pd.concat([existing2, pd.DataFrame(sc_results)], ignore_index=True)


In [20]:
# Disabled: legacy per-account cell retained from parallel runs.
print('Skipping legacy account-specific cell 17; use Cells 5–11 for the full 50-problem run.')



MessageError: Error: credential propagation was unsuccessful